# NN02: 64/32 with validation-selected rates

Executable notebook copy of `Models/NN02_model.py`. The class definition below is copied from that file and uses the shared NN01 training implementation. Ten seeds, SGD, cross-entropy, whole-election early stopping, patience 20. Set `TRAIN_MODEL=True` to fit the 2019 development example; by default the notebook displays the saved seven-election results.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Models/NN01_model.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PACKAGE = ROOT / 'Analysis and model development/05_nn_first_development'
RESULTS = PACKAGE / 'results'


## Model definition

In [2]:
"""NN02: 64/32 validation-selected rate. Ten seeds; latest whole election held out; patience 20.

Uses the shared implementation in NN01_model without changing its configuration.
See Analysis and model development/05_nn_first_development/.
"""
from Models.NN01_model import NeuralNetworkModel as BaseNeuralNetworkModel

HIDDEN_SIZES = (64, 32)
LEARNING_RATES = (0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)
CLASS_LABELS = ('con', 'lab', 'lib', 'natSW', 'oth')


class NeuralNetworkModel(BaseNeuralNetworkModel):
    def __init__(self):
        super().__init__(hidden_sizes=HIDDEN_SIZES, learning_rates=LEARNING_RATES,
                         name='NN02: 64/32 validation-selected rate', class_labels=CLASS_LABELS)


In [3]:
model = NeuralNetworkModel()
print(model.name)
print('Hidden layers:', model.hidden_sizes)
print('Learning rates:', model.learning_rates)
print('Classes:', model.class_labels)

NN02: 64/32 validation-selected rate
Hidden layers: (64, 32)
Learning rates: (0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)
Classes: ('con', 'lab', 'lib', 'natSW', 'oth')


## Saved historical development results

In [4]:
historical=pd.read_csv(RESULTS/'election_diagnostics.csv')
historical=historical.loc[(historical.architecture=='64_32') & (historical.policy=='validation_all_rates')]
display(historical[['evaluation_year','learning_rate','accuracy','changed_accuracy','overall_changed_score','log_loss','correct_changes','false_changes_on_held_seats']])

,evaluation_year,learning_rate,accuracy,changed_accuracy,overall_changed_score,log_loss,correct_changes,false_changes_on_held_seats
6,1997,1.0,0.684867,0.056250,0.370559,1.172363,9,5
14,2001,1.0,0.945398,0.041667,0.493532,0.198077,1,12
22,2005,0.2,0.791401,0.631579,0.711490,0.459736,36,110
30,2010,1.0,0.892405,0.830357,0.861381,0.305171,93,49
38,2015,1.0,0.759494,0.137615,0.448554,0.954890,15,58
46,2017,1.0,0.895570,0.089552,0.492561,0.403865,6,5
54,2019,0.3,0.897152,0.342105,0.619629,0.236477,26,15


## Optional fit: train through 2015, validate on 2017, evaluate on 2019
Training uses the retained historical snapshot only. The whole 2017 election selects checkpoints; evaluation outcomes are not used to choose the rate.

In [5]:
TRAIN_MODEL = False
if TRAIN_MODEL:
    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, log_loss
    torch.set_num_threads(1)
    data=pd.read_csv(PACKAGE/'data/train.csv')
    training=data.loc[data.election<=2017]
    evaluation=data.loc[data.election==2019]
    model.train(training)
    probabilities=model.predict_proba(evaluation)
    predictions=model.predict(evaluation)
    changed=evaluation.previous_winner.notna() & evaluation.winner.ne(evaluation.previous_winner)
    print('Selected rate:',model.selected_learning_rate)
    print('Accuracy:',accuracy_score(evaluation.winner,predictions))
    print('Changed-seat accuracy:',accuracy_score(evaluation.loc[changed,'winner'],predictions[changed]))
    print('Log loss:',log_loss(evaluation.winner,probabilities,labels=model.classes_))
    display(model.learning_rate_summary)
else:
    print('Saved results displayed. Set TRAIN_MODEL=True to run the full ten-seed procedure.')

Saved results displayed. Set TRAIN_MODEL=True to run the full ten-seed procedure.
